In [1]:
from dotenv import load_dotenv
from pprint import pprint
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

load_dotenv()

True

## Summarize messages

In [3]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [48]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "12"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, the capital of the moon, including its weather and demographic details.\n\n## SUMMARY\nThe capital of the moon is Lunapolis. The weather in Lunapolis is clear, with a high of 120°C and a low of -100°C. There are 100,000 cheese miners living in Lunapolis. It is anticipated that the cheese miners' union will strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='bafdd59e-d237-4ea9-87f4-58d00ec49b4c'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='d18e7da4-4fa4-47ea-bb9f-9616735c6035'),
              AIMessage(content='If I were Lunapolis’ new president, I’d respond to the ch

In [41]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, a fictional location on the moon, including its capital, weather, population of cheese miners, and the state of their union.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is clear, with a high temperature of 120C and a low of -100C.
- There are 100,000 cheese miners living in Lunapolis.
- It is suggested that the cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [4]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [6]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='539d3db7-ef1d-41a4-a66f-8b6c5d65432c'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='2bfdb0a3-18cb-4160-a8ab-55b508cc50b4', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='31123808-e691-4e17-9351-42582da2daff'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='11cbf3e0-1ba3-4ce9-b3a7-ef6b83b12c25', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='0a6d0e94-7af7-40df-8298-20a1d07a472d'),
              AIMessage(content='I can’t read your device’s temperature from here. If you want 

In [7]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If you want the exact temp, tell me the device type (e.g., Windows PC, Mac, Android phone, iPhone, router) and the model. I’ll give you precise steps. In the meantime:

- If the device is hot to the touch: power off, unplug, and let it cool in a well-ventilated area. Check that vents aren’t blocked and the fans are running.

- How to check temperature (typical options by device):
  - Windows PC: use a free tool like HWMonitor or Core Temp to read CPU/GPU temps. Idle temps around 30–50°C; safe under load usually up to 70–85°C (check your CPU/GPU specs).
  - Mac: apps like iStat Menus or Macs Fan Control show temps; there isn’t a built-in simple temp readout in macOS.
  - Android: Settings > Battery > Temperature (if available) or use an app like CPU-Z/AIDA64 to read CPU temp.
  - iPhone: iOS doesn’t provide a standard temperature readout; third-party apps may offer limited readings, but it’s less straightforward.
  - Other devices (route